# Furnace v3 - Telemetry + Labels Generation


In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
from tqdm import tqdm


In [3]:

class FurnaceTwin:

    def __init__(self, asset_id, failure_scenario):
        self.asset_id = asset_id
        self.failure_scenario = failure_scenario
        self.health = 100.0
        self.burner_degradation = 0
        self.tube_overheating = 0
        self.combustion_imbalance = 0

    def update_health(self):

        wear = np.random.uniform(0.00015,0.0008)

        if self.failure_scenario == "Burner Failure":
            self.burner_degradation += wear * 5
            self.health -= wear * 1.8

        elif self.failure_scenario == "Tube Overheating":
            self.tube_overheating += wear * 5
            self.health -= wear * 1.6

        elif self.failure_scenario == "Combustion Imbalance":
            self.combustion_imbalance += wear * 5
            self.health -= wear * 1.4

        else:
            self.health -= wear * 0.3

        self.health = max(self.health,0)

    def get_rul(self):
        return round((self.health/100)*300,2)

    def get_failure_next_30_days(self):
        return int(self.get_rul() <= 30)

    def generate_telemetry(self):

        self.update_health()

        furnace_temp = 850 + self.combustion_imbalance*3 + np.random.normal(0,5)
        tube_skin_temp = 450 + self.tube_overheating*5 + np.random.normal(0,3)
        fuel_gas_flow = 1000 + self.burner_degradation*4 + np.random.normal(0,10)
        oxygen_level = 3 + self.combustion_imbalance*0.2 + np.random.normal(0,0.1)
        stack_temp = 250 + self.burner_degradation*2 + np.random.normal(0,2)
        burner_efficiency = max(60,95-self.burner_degradation*2)

        telemetry = {
            "asset_id": self.asset_id,
            "furnace_temp": round(furnace_temp,2),
            "tube_skin_temp": round(tube_skin_temp,2),
            "fuel_gas_flow": round(fuel_gas_flow,2),
            "oxygen_level": round(oxygen_level,2),
            "stack_temp": round(stack_temp,2),
            "burner_efficiency": round(burner_efficiency,2)
        }

        labels = {
            "asset_id": self.asset_id,
            "failure_mode": self.failure_scenario,
            "rul_days": self.get_rul(),
            "failure_next_30_days": self.get_failure_next_30_days()
        }

        return telemetry, labels


In [4]:

assets = [
    FurnaceTwin("H-501","Burner Failure"),
    FurnaceTwin("H-502","Tube Overheating"),
    FurnaceTwin("H-503","Combustion Imbalance"),
    FurnaceTwin("H-504","Tube Overheating"),
    FurnaceTwin("H-505","Healthy")
]


In [5]:

telemetry_records=[]
label_records=[]

minutes=90*24*60
start_time=datetime.now()

for minute in tqdm(range(minutes)):
    current_time=start_time+timedelta(minutes=minute)

    for asset in assets:

        telemetry,labels=asset.generate_telemetry()

        telemetry["timestamp"]=current_time
        labels["timestamp"]=current_time

        telemetry_records.append(telemetry)
        label_records.append(labels)

telemetry_df=pd.DataFrame(telemetry_records)
labels_df=pd.DataFrame(label_records)

telemetry_df.to_csv("furnace_telemetry.csv",index=False)
labels_df.to_csv("furnace_labels.csv",index=False)

print(telemetry_df.shape)
print(labels_df.shape)


100%|██████████| 129600/129600 [00:10<00:00, 12626.63it/s]


(648000, 8)
(648000, 5)
